In [10]:
from google.colab import drive
drive.mount("/content/drive")

%cd /content/drive/MyDrive/katabatic1
!ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/katabatic1
 CONTRIBUTING.md		       main.py		        README.md
'C:\Users\Prabu\Downloads\Katabatic'   Makefile		        Results
 dev_deps.py			       medgan_adult.py	        runs
 discretized_data		       MODEL_CONTRIBUTIONS.md   sample_data
 encoded_data			       outputs		        scripts
 example.ipynb			       poetry.lock	        synthetic
 examples			       __pycache__	        utils.py
 katabatic			       pyproject.toml	        venv
 LICENSE			       raw_data


In [11]:
!ls katabatic/


cli  evaluate  __init__.py  models  pipeline  __pycache__  utils


In [12]:
!ls katabatic/models


arf	       CopulaGAN	ganblr	     maf     pategan	  tabddpm
base_model.py  ctgan		great	     medgan  __pycache__  tabpfgen
codi	       forestdiffusion	__init__.py  meg     registry.py  tabsyn


In [5]:
pip install sdv


In [6]:
import katabatic.models.CopulaGAN
print("CopulaGAN module found ✅")


CopulaGAN module found ✅


In [7]:
pip install sdv xgboost


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import time
import pandas as pd
from pathlib import Path

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.models.CopulaGAN.adapter import CopulaGANAdapter
from katabatic.evaluate.tstr.evaluation import TSTREvaluation
from sklearn.preprocessing import OrdinalEncoder

ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV = ROOT / "raw_data" / "car.csv"
SAMPLE_DIR = ROOT / "sample_data" / "car"
SYNTH_DIR = ROOT / "synthetic" / "car" / "copulagan"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()

pipeline = TrainTestSplitPipeline(
    model=lambda: CopulaGANAdapter(target_col="target")
)

pipeline.run(
    input_csv=str(RAW_CSV),
    output_dir=str(SAMPLE_DIR),
    label_col="6"
)

cg = CopulaGANAdapter(target_col="target")
cg.train(output_dir=str(SAMPLE_DIR), label_col="6")

n_samples = len(pd.read_csv(SAMPLE_DIR / "x_train.csv"))
x_synth, y_synth = cg.sample(n_samples)

pd.DataFrame(x_synth).to_csv(SYNTH_DIR / "x_synth.csv", index=False)
pd.DataFrame({"6": y_synth}).to_csv(SYNTH_DIR / "y_synth.csv", index=False)

x_train_df = pd.read_csv(SAMPLE_DIR / "x_train.csv")
x_test_df = pd.read_csv(SAMPLE_DIR / "x_test.csv")
x_synth_df = pd.read_csv(SYNTH_DIR / "x_synth.csv")

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

encoder.fit(x_train_df.astype(str))

x_train_enc = encoder.transform(x_train_df.astype(str))
x_test_enc = encoder.transform(x_test_df.astype(str))
x_synth_enc = encoder.transform(x_synth_df.astype(str))

pd.DataFrame(x_train_enc).to_csv(SAMPLE_DIR / "x_train.csv", index=False)
pd.DataFrame(x_test_enc).to_csv(SAMPLE_DIR / "x_test.csv", index=False)
pd.DataFrame(x_synth_enc).to_csv(SYNTH_DIR / "x_synth.csv", index=False)

print("All features encoded numerically for TSTR")

x_test = pd.read_csv(SAMPLE_DIR / "x_test.csv")
x_test.columns = pd.read_csv(SYNTH_DIR / "x_synth.csv").columns
x_test.to_csv(SAMPLE_DIR / "x_test.csv", index=False)

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(SAMPLE_DIR)
)

results = tstr.evaluate()
print(results)

print("Total runtime (minutes):", round((time.time() - t0) / 60, 2))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded data with shape: (1728, 7)
Saved train/test full data
Train size: (1382, 7), Test size: (346, 7)
Train label distribution:
 6
unacc    0.700434
acc      0.222142
good     0.039797
vgood    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
unacc    0.699422
acc      0.222543
good     0.040462
vgood    0.037572
Name: proportion, dtype: float64
Saved X/y split
Training shape: (1382, 6) (1382,)
Test shape: (346, 6) (346,)


/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:168: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:168: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


All features encoded numerically for TSTR


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [03:30:55] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results/car/copulagan_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.6994
F1 Score: 0.5757

MLP:
Accuracy: 0.6705
F1 Score: 0.5752

RF:
Accuracy: 0.5925
F1 Score: 0.5707

XGBoost:
Accuracy: 0.5723
F1 Score: 0.5694
{'LR': {'Accuracy': 0.6994219653179191, 'F1 Score': 0.5757146789351579}, 'MLP': {'Accuracy': 0.6705202312138728, 'F1 Score': 0.5752123556447929}, 'RF': {'Accuracy': 0.5924855491329479, 'F1 Score': 0.5706868412104726}, 'XGBoost': {'Accuracy': 0.5722543352601156, 'F1 Score': 0.569414493079552}}
Total runtime (minutes): 0.59


In [14]:
from google.colab import drive
drive.mount("/content/drive")

import time
import warnings
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.models.CopulaGAN.adapter import CopulaGANAdapter
from katabatic.evaluate.tstr.evaluation import TSTREvaluation

warnings.filterwarnings("ignore", message="The 'SingleTableMetadata' is deprecated")
warnings.filterwarnings("ignore", message="We strongly recommend saving the metadata")

ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV = ROOT / "raw_data" / "magic.csv"
SAMPLE_DIR = ROOT / "sample_data" / "magic"
SYNTH_DIR = ROOT / "synthetic" / "magic" / "copulagan"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()

pipeline = TrainTestSplitPipeline(
    model=lambda: CopulaGANAdapter()
)

pipeline.run(
    input_csv=str(RAW_CSV),
    output_dir=str(SAMPLE_DIR)
)

cg = CopulaGANAdapter()
cg.train(output_dir=str(SAMPLE_DIR))

x_train_df = pd.read_csv(SAMPLE_DIR / "x_train.csv")
n_samples = len(x_train_df)

x_synth, y_synth = cg.sample(n_samples)

pd.DataFrame(
    x_synth,
    columns=x_train_df.columns
).to_csv(SYNTH_DIR / "x_synth.csv", index=False)

pd.DataFrame(
    {"class": y_synth}
).to_csv(SYNTH_DIR / "y_synth.csv", index=False)

x_test_df = pd.read_csv(SAMPLE_DIR / "x_test.csv")
x_synth_df = pd.read_csv(SYNTH_DIR / "x_synth.csv")

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
encoder.fit(x_train_df.astype(str))

pd.DataFrame(
    encoder.transform(x_train_df.astype(str)),
    columns=x_train_df.columns
).to_csv(SAMPLE_DIR / "x_train.csv", index=False)

pd.DataFrame(
    encoder.transform(x_test_df.astype(str)),
    columns=x_test_df.columns
).to_csv(SAMPLE_DIR / "x_test.csv", index=False)

pd.DataFrame(
    encoder.transform(x_synth_df.astype(str)),
    columns=x_synth_df.columns
).to_csv(SYNTH_DIR / "x_synth.csv", index=False)

x_test = pd.read_csv(SAMPLE_DIR / "x_test.csv")
x_test.columns = pd.read_csv(SYNTH_DIR / "x_synth.csv").columns
x_test.to_csv(SAMPLE_DIR / "x_test.csv", index=False)

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(SAMPLE_DIR)
)

results = tstr.evaluate()

print(results)
print("Total runtime (minutes):", round((time.time() - t0) / 60, 2))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded data with shape: (19020, 11)
Saved train/test full data
Train size: (15216, 11), Test size: (3804, 11)
Train label distribution:
 class
g    0.648396
h    0.351604
Name: proportion, dtype: float64
Test label distribution:
 class
g    0.648265
h    0.351735
Name: proportion, dtype: float64
Saved X/y split
Training shape: (15216, 10) (15216,)
Test shape: (3804, 10) (3804,)


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



Results saved to: Results/magic/copulagan_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7963
F1 Score: 0.7984
AUC: 0.8631

MLP:
Accuracy: 0.8289
F1 Score: 0.8280
AUC: 0.8696

RF:
Accuracy: 0.7763
F1 Score: 0.7785
AUC: 0.8369

XGBoost:
Accuracy: 0.8141
F1 Score: 0.8142
AUC: 0.8644
{'LR': {'Accuracy': 0.796267087276551, 'F1 Score': 0.7984174721711123, 'AUC': np.float64(0.8630614018817352)}, 'MLP': {'Accuracy': 0.8288643533123028, 'F1 Score': 0.8279796764194332, 'AUC': np.float64(0.8696219254507035)}, 'RF': {'Accuracy': 0.7762881177707676, 'F1 Score': 0.778539892442335, 'AUC': np.float64(0.8368788619394165)}, 'XGBoost': {'Accuracy': 0.814143007360673, 'F1 Score': 0.8142220113987917, 'AUC': np.float64(0.8643582618984406)}}
Total runtime (minutes): 8.26


In [15]:
from google.colab import drive
drive.mount("/content/drive")

import time
import warnings
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.models.CopulaGAN.adapter import CopulaGANAdapter
from katabatic.evaluate.tstr.evaluation import TSTREvaluation

warnings.filterwarnings("ignore", message="The 'SingleTableMetadata' is deprecated")
warnings.filterwarnings("ignore", message="We strongly recommend saving the metadata")

ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV = ROOT / "raw_data" / "nursery.csv"
SAMPLE_DIR = ROOT / "sample_data" / "nursery"
SYNTH_DIR = ROOT / "synthetic" / "nursery" / "copulagan"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()

pipeline = TrainTestSplitPipeline(
    model=lambda: CopulaGANAdapter()
)

pipeline.run(
    input_csv=str(RAW_CSV),
    output_dir=str(SAMPLE_DIR)
)

cg = CopulaGANAdapter()
cg.train(output_dir=str(SAMPLE_DIR))

x_train_df = pd.read_csv(SAMPLE_DIR / "x_train.csv")
n_samples = len(x_train_df)

x_synth, y_synth = cg.sample(n_samples)

pd.DataFrame(
    x_synth,
    columns=x_train_df.columns
).to_csv(SYNTH_DIR / "x_synth.csv", index=False)

pd.DataFrame(
    {"target": y_synth}
).to_csv(SYNTH_DIR / "y_synth.csv", index=False)

x_test_df = pd.read_csv(SAMPLE_DIR / "x_test.csv")
x_synth_df = pd.read_csv(SYNTH_DIR / "x_synth.csv")

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
encoder.fit(x_train_df.astype(str))

pd.DataFrame(
    encoder.transform(x_train_df.astype(str)),
    columns=x_train_df.columns
).to_csv(SAMPLE_DIR / "x_train.csv", index=False)

pd.DataFrame(
    encoder.transform(x_test_df.astype(str)),
    columns=x_test_df.columns
).to_csv(SAMPLE_DIR / "x_test.csv", index=False)

pd.DataFrame(
    encoder.transform(x_synth_df.astype(str)),
    columns=x_synth_df.columns
).to_csv(SYNTH_DIR / "x_synth.csv", index=False)

x_test = pd.read_csv(SAMPLE_DIR / "x_test.csv")
x_test.columns = pd.read_csv(SYNTH_DIR / "x_synth.csv").columns
x_test.to_csv(SAMPLE_DIR / "x_test.csv", index=False)

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(SAMPLE_DIR)
)

results = tstr.evaluate()

print(results)
print("Total runtime (minutes):", round((time.time() - t0) / 60, 2))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded data with shape: (12960, 9)
Saved train/test full data
Train size: (10368, 9), Test size: (2592, 9)
Train label distribution:
 8
not_recom     0.333333
priority      0.329186
spec_prior    0.312018
very_recom    0.025270
recommend     0.000193
Name: proportion, dtype: float64
Test label distribution:
 8
not_recom     0.333333
priority      0.329090
spec_prior    0.312114
very_recom    0.025463
Name: proportion, dtype: float64
Saved X/y split
Training shape: (10368, 8) (10368,)
Test shape: (2592, 8) (2592,)


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [04:26:55] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results/nursery/copulagan_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7079
F1 Score: 0.6527

MLP:
Accuracy: 0.8438
F1 Score: 0.8313

RF:
Accuracy: 0.7762
F1 Score: 0.7703

XGBoost:
Accuracy: 0.8106
F1 Score: 0.8034
{'LR': {'Accuracy': 0.7079475308641975, 'F1 Score': 0.6527102771427491}, 'MLP': {'Accuracy': 0.84375, 'F1 Score': 0.8312533983428562}, 'RF': {'Accuracy': 0.7762345679012346, 'F1 Score': 0.770278562482657}, 'XGBoost': {'Accuracy': 0.810570987654321, 'F1 Score': 0.8033746410461415}}
Total runtime (minutes): 5.2


In [16]:
from google.colab import drive
drive.mount("/content/drive")

import time
import warnings
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.models.CopulaGAN.adapter import CopulaGANAdapter
from katabatic.evaluate.tstr.evaluation import TSTREvaluation

warnings.filterwarnings("ignore", message="The 'SingleTableMetadata' is deprecated")
warnings.filterwarnings("ignore", message="We strongly recommend saving the metadata")

ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV = ROOT / "raw_data" / "adult.csv"
SAMPLE_DIR = ROOT / "sample_data" / "adult"
SYNTH_DIR = ROOT / "synthetic" / "adult" / "copulagan"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()

pipeline = TrainTestSplitPipeline(
    model=lambda: CopulaGANAdapter()
)

pipeline.run(
    input_csv=str(RAW_CSV),
    output_dir=str(SAMPLE_DIR)
)

cg = CopulaGANAdapter()
cg.train(output_dir=str(SAMPLE_DIR))

x_train_df = pd.read_csv(SAMPLE_DIR / "x_train.csv")
n_samples = len(x_train_df)

x_synth, y_synth = cg.sample(n_samples)

pd.DataFrame(
    x_synth,
    columns=x_train_df.columns
).to_csv(SYNTH_DIR / "x_synth.csv", index=False)

pd.DataFrame(
    {"income": y_synth}
).to_csv(SYNTH_DIR / "y_synth.csv", index=False)

x_test_df = pd.read_csv(SAMPLE_DIR / "x_test.csv")
x_synth_df = pd.read_csv(SYNTH_DIR / "x_synth.csv")

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
encoder.fit(x_train_df.astype(str))

pd.DataFrame(
    encoder.transform(x_train_df.astype(str)),
    columns=x_train_df.columns
).to_csv(SAMPLE_DIR / "x_train.csv", index=False)

pd.DataFrame(
    encoder.transform(x_test_df.astype(str)),
    columns=x_test_df.columns
).to_csv(SAMPLE_DIR / "x_test.csv", index=False)

pd.DataFrame(
    encoder.transform(x_synth_df.astype(str)),
    columns=x_synth_df.columns
).to_csv(SYNTH_DIR / "x_synth.csv", index=False)

x_test = pd.read_csv(SAMPLE_DIR / "x_test.csv")
x_test.columns = pd.read_csv(SYNTH_DIR / "x_synth.csv").columns
x_test.to_csv(SAMPLE_DIR / "x_test.csv", index=False)

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(SAMPLE_DIR)
)

results = tstr.evaluate()

print(results)
print("Total runtime (minutes):", round((time.time() - t0) / 60, 2))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
<=50K    0.759175
>50K     0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
<=50K    0.759251
>50K     0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



Results saved to: Results/adult/copulagan_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7427
F1 Score: 0.7498
AUC: 0.7620

MLP:
Accuracy: 0.7657
F1 Score: 0.7771
AUC: 0.8132

RF:
Accuracy: 0.7818
F1 Score: 0.7922
AUC: 0.8493

XGBoost:
Accuracy: 0.7485
F1 Score: 0.7658
AUC: 0.8572
{'LR': {'Accuracy': 0.742668509135575, 'F1 Score': 0.7497549706344289, 'AUC': np.float64(0.7619547161635131)}, 'MLP': {'Accuracy': 0.7656993704897896, 'F1 Score': 0.7771009773723622, 'AUC': np.float64(0.8132486690328304)}, 'RF': {'Accuracy': 0.7818209734377399, 'F1 Score': 0.7922348505370657, 'AUC': np.float64(0.8492501057551433)}, 'XGBoost': {'Accuracy': 0.7485029940119761, 'F1 Score': 0.7658372550178899, 'AUC': np.float64(0.8572067616227481)}}
Total runtime (minutes): 17.62


In [17]:
from google.colab import drive
drive.mount("/content/drive")

import time
import warnings
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.models.CopulaGAN.adapter import CopulaGANAdapter
from katabatic.evaluate.tstr.evaluation import TSTREvaluation

warnings.filterwarnings("ignore", message="The 'SingleTableMetadata' is deprecated")
warnings.filterwarnings("ignore", message="We strongly recommend saving the metadata")

ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV = ROOT / "raw_data" / "shuttle.csv"
SAMPLE_DIR = ROOT / "sample_data" / "shuttle"
SYNTH_DIR = ROOT / "synthetic" / "shuttle" / "copulagan"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()

pipeline = TrainTestSplitPipeline(
    model=lambda: CopulaGANAdapter()
)

pipeline.run(
    input_csv=str(RAW_CSV),
    output_dir=str(SAMPLE_DIR)
)

cg = CopulaGANAdapter()
cg.train(output_dir=str(SAMPLE_DIR))

x_train_df = pd.read_csv(SAMPLE_DIR / "x_train.csv")
n_samples = len(x_train_df)

x_synth, y_synth = cg.sample(n_samples)

pd.DataFrame(
    x_synth,
    columns=x_train_df.columns
).to_csv(SYNTH_DIR / "x_synth.csv", index=False)

pd.DataFrame(
    {"target": y_synth}
).to_csv(SYNTH_DIR / "y_synth.csv", index=False)

x_test_df = pd.read_csv(SAMPLE_DIR / "x_test.csv")
x_synth_df = pd.read_csv(SYNTH_DIR / "x_synth.csv")

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
encoder.fit(x_train_df.astype(str))

pd.DataFrame(
    encoder.transform(x_train_df.astype(str)),
    columns=x_train_df.columns
).to_csv(SAMPLE_DIR / "x_train.csv", index=False)

pd.DataFrame(
    encoder.transform(x_test_df.astype(str)),
    columns=x_test_df.columns
).to_csv(SAMPLE_DIR / "x_test.csv", index=False)

pd.DataFrame(
    encoder.transform(x_synth_df.astype(str)),
    columns=x_synth_df.columns
).to_csv(SYNTH_DIR / "x_synth.csv", index=False)

x_test = pd.read_csv(SAMPLE_DIR / "x_test.csv")
x_test.columns = pd.read_csv(SYNTH_DIR / "x_synth.csv").columns
x_test.to_csv(SAMPLE_DIR / "x_test.csv", index=False)

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(SAMPLE_DIR)
)

results = tstr.evaluate()

print(results)
print("Total runtime (minutes):", round((time.time() - t0) / 60, 2))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded data with shape: (58000, 10)
Saved train/test full data
Train size: (46400, 10), Test size: (11600, 10)
Train label distribution:
 class
1    0.785970
4    0.153491
5    0.056336
3    0.002953
2    0.000862
7    0.000216
6    0.000172
Name: proportion, dtype: float64
Test label distribution:
 class
1    0.785948
4    0.153534
5    0.056293
3    0.002931
2    0.000862
7    0.000259
6    0.000172
Name: proportion, dtype: float64
Saved X/y split
Training shape: (46400, 9) (46400,)
Test shape: (11600, 9) (11600,)


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [05:14:36] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results/shuttle/copulagan_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.9091
F1 Score: 0.9198

MLP:
Accuracy: 0.9875
F1 Score: 0.9914

RF:
Accuracy: 0.9816
F1 Score: 0.9862

XGBoost:
Accuracy: 0.9861
F1 Score: 0.9898
{'LR': {'Accuracy': 0.909051724137931, 'F1 Score': 0.919757595175565}, 'MLP': {'Accuracy': 0.9875, 'F1 Score': 0.9914135008966041}, 'RF': {'Accuracy': 0.981551724137931, 'F1 Score': 0.9862008796448011}, 'XGBoost': {'Accuracy': 0.9861206896551724, 'F1 Score': 0.989755115962127}}
Total runtime (minutes): 26.43
